In [5]:
import pandas as pd
import anndata as ad

input_path = "test-data-TOO/GTEx_v8_all_tissues_all_samples_rawCounts.tsv"
output_path = "test-data-TOO/GTEx_v8_rawCounts_GeneSymbol.h5ad"

df = pd.read_csv(input_path, sep="\t", header=None, low_memory=False)

# Strip leading/trailing whitespace
tissues = df.iloc[0, 2:].astype(str).str.strip().values
sample_ids = df.iloc[1, 2:].astype(str).str.strip().values

gene_symbols = df.iloc[2:, 1].astype(str).str.strip().values
ensembl_ids = df.iloc[2:, 0].astype(str).str.strip().values

X = df.iloc[2:, 2:].astype("float32")
X.index = gene_symbols
X.columns = sample_ids

adata = ad.AnnData(X=X.T.values)

adata.obs_names = sample_ids
adata.obs["MergedClusters"] = tissues

adata.var_names = gene_symbols
adata.var["EnsemblID"] = ensembl_ids
adata.var_names_make_unique()

# ---- Verification ----
counts = adata.obs["MergedClusters"].value_counts().sort_index()

print(f"\nNumber of tissue categories: {len(counts)}\n")
print("Tissue categories and sample counts:")
for tissue, count in counts.items():
    print(f"{repr(tissue):30} {count}")

# Extra safeguard: detect any remaining leading/trailing whitespace
problematic = [
    t for t in adata.obs["MergedClusters"].unique()
    if t != t.strip()
]

if problematic:
    print("\nWARNING: Labels with leading/trailing whitespace found:")
    for t in problematic:
        print(repr(t))
else:
    print("\n✓ No labels with leading/trailing whitespace detected.")
# ----------------------

adata.write_h5ad(output_path)

print("\n", adata)
print(adata.obs.head())
print(adata.var.head())

/exports/igmm/eddie/wendy-lab/Alexis/anaconda/envs/jupyter2/lib/python3.10/site-packages/anndata/utils.py:261: UserWarning: Suffix used (-[0-9]+) to deduplicate index values may make index values difficult to interpret. There values with a similar suffixes in the index. Consider using a different delimiter by passing `join={delimiter}`Example key collisions generated by the make_index_unique algorithm: ['SNORD116-1', 'SNORD116-2', 'SNORD116-3', 'SNORD116-5', 'SNORD116-6']
  warnings.warn(



Number of tissue categories: 30

Tissue categories and sample counts:
'Adipose'                      1204
'Adrenal_gland'                258
'Arteries'                     1335
'Bladder'                      21
'Brain'                        2642
'Breast'                       459
'ColonSigmoid'                 373
'ColonTransverse'              406
'Esophagus'                    890
'EsophagusMucosa'              555
'FemaleReproductive'           506
'Fibroblasts'                  504
'Heart'                        861
'Kidney'                       89
'Liver'                        226
'Lung'                         578
'Lymphocytes'                  174
'MuscleSkeletal'               803
'NerveTibial'                  619
'Pancreas'                     328
'Pituitary'                    283
'Prostate'                     245
'SalivaryGland'                162
'Skin'                         1305
'SmallIntestine'               187
'Spleen'                       241
'Stomach'        

In [6]:
import anndata as ad

adata = ad.read_h5ad("test-data-TOO/GTEx_v8_rawCounts_GeneSymbol.h5ad")

counts = adata.obs["MergedClusters"].value_counts(dropna=False)

for tissue, count in counts.items():
    print(f"{repr(tissue):20} {count}")

'Brain'              2642
'Arteries'           1335
'Skin'               1305
'Adipose'            1204
'Esophagus'          890
'Heart'              861
'MuscleSkeletal'     803
'Whole_blood'        755
'Thyroid'            653
'NerveTibial'        619
'Lung'               578
'EsophagusMucosa'    555
'FemaleReproductive' 506
'Fibroblasts'        504
'Breast'             459
'ColonTransverse'    406
'ColonSigmoid'       373
'Testis'             361
'Stomach'            359
'Pancreas'           328
'Pituitary'          283
'Adrenal_gland'      258
'Prostate'           245
'Spleen'             241
'Liver'              226
'SmallIntestine'     187
'Lymphocytes'        174
'SalivaryGland'      162
'Kidney'             89
'Bladder'            21


In [1]:
# Script to rename the GTEx gene names to match the gene names of the gencode used for tissue mixture generation
import pandas as pd
import anndata as ad

gtex_input = "test-data-TOO/GTEx_v8_all_tissues_all_samples_rawCounts.tsv"
training_h5ad = "/exports/eddie/scratch/s2556897/DECODE/test-data-TOO/GTEx_v8_rawCounts_GeneSymbol.h5ad"

output_file = "GTEx_v8_all_tissues_all_samples_TrainingGeneNames.tsv"

# Load only EnsemblID and original GeneSymbol from the GTEx raw counts file
gtex_genes = pd.read_csv(
    gtex_input,
    sep="\t",
    usecols=[0, 1],
    dtype=str
)

gtex_genes.columns = ["EnsemblID", "GeneSymbol"]

# Remove the sample-name/header-like row
gtex_genes = gtex_genes.dropna(subset=["EnsemblID"])
gtex_genes = gtex_genes[gtex_genes["EnsemblID"].str.startswith("ENSG")].copy()

# Load exact training gene names from the h5ad used by DECODE
adata_ref = ad.read_h5ad(training_h5ad)
train_genes = adata_ref.var_names.astype(str).tolist()

# Critical safety check: order must match
if len(gtex_genes) != len(train_genes):
    raise ValueError(
        f"Length mismatch: GTEx gene rows = {len(gtex_genes)}, "
        f"h5ad var_names = {len(train_genes)}. "
        "Cannot safely assign TrainingGeneName by order."
    )

# Since the h5ad was generated from this GTEx file, assign training names by order
gtex_genes["TrainingGeneName"] = train_genes

out = gtex_genes[["EnsemblID", "GeneSymbol", "TrainingGeneName"]]

out.to_csv(output_file, sep="\t", index=False)

print(f"Saved: {output_file}")
print(f"Genes written: {len(out)}")
print(f"Changed names: {(out['GeneSymbol'] != out['TrainingGeneName']).sum()}")

# Useful sanity checks
for ens in ["ENSG00000238009", "ENSG00000239945", "ENSG00000290826"]:
    ens_clean = out["EnsemblID"].str.split(".").str[0]
    hit = out[ens_clean == ens]

    if len(hit):
        print(hit.to_string(index=False))
    else:
        print(f"{ens}: NOT_FOUND")

Saved: GTEx_v8_all_tissues_all_samples_TrainingGeneNames.tsv
Genes written: 56200
Changed names: 1608
        EnsemblID   GeneSymbol TrainingGeneName
ENSG00000238009.6 RP11-34P13.7     RP11-34P13.7
ENSG00000239945: NOT_FOUND
ENSG00000290826: NOT_FOUND
